# Fine-tune the retrieval encoder (Colab GPU)

Adapts the pre-trained Sentence-BERT model to the course catalog, then compares
retrieval metrics **before vs after**.

**Runtime → Change runtime type → GPU (T4)** before running.

Steps: install → build base artifacts → generate pairs → fine-tune → evaluate both → download the model.

In [ ]:
# 1. Get the code + deps
!git clone https://github.com/KhangPhanTZ/course-recommender-simple.git
%cd course-recommender-simple
!pip install -q -r requirements.txt -r requirements-train.txt

In [ ]:
# 2. Build BASE artifacts (pre-trained encoder) from the committed catalog
!python -m src.pipeline --mode build --data data/
!python scripts/run_eval.py   # baseline metrics

In [ ]:
# 3. Generate training pairs
#    proxy = offline, no key.  llm = synthetic queries (set the key below first).
# import os; os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'   # for --mode llm
!python scripts/generate_pairs.py --mode proxy --out data/finetune/pairs.jsonl
!wc -l data/finetune/pairs.jsonl

In [ ]:
# 4. Fine-tune the bi-encoder on the pairs
!python -m src.training.finetune \
    --pairs data/finetune/pairs.jsonl \
    --base BAAI/bge-small-en-v1.5 \
    --out artifacts/finetuned-encoder \
    --epochs 1 --batch 32

In [ ]:
# 5. Rebuild with the FINE-TUNED encoder and re-evaluate -> AFTER metrics
import yaml
cfg = yaml.safe_load(open('config/config.yaml'))
cfg['sbert_model'] = 'artifacts/finetuned-encoder'
yaml.safe_dump(cfg, open('config/config.yaml', 'w'), sort_keys=False)
!python -m src.pipeline --mode build --data data/
!python scripts/run_eval.py   # compare these numbers to the baseline in step 2

In [ ]:
# 6. Download the fine-tuned model (or push it to S3 / Hugging Face)
!zip -rq finetuned-encoder.zip artifacts/finetuned-encoder
from google.colab import files
files.download('finetuned-encoder.zip')